# Day 1 — Foundations & Workspace

This notebook covers: **PySpark DataFrames**, **Spark SQL**, and **Widgets**.

---

## Databricks Account — Use Your Student ID

All students use the **same Databricks account**. Use your **student ID** in folder and cluster names:

- **Student IDs:** `u01` through `u16` (instructor will assign).
- **Folder name:** `day01-uXX` (e.g. `day01-u05`).
- **Cluster name:** `day01-cluster-uXX` (e.g. `day01-cluster-u05`).

Attach this notebook to **your** cluster: `day01-cluster-uXX`.

---

## Upload data to File Store

**Day 1 uses File Store only** (no DBFS mount; mounting is covered on Day 2).

1. **Get the data** from the course repo folder **`data/flight-data/`** (contains `csv/` and `json/` with flight summary files).
2. In Databricks: go to **Data** in the left sidebar → **Add Data** → **Upload File** (or drag files into the upload area). Upload the contents so you have:
   - CSV: `.../flight-data/csv/2010-summary.csv` and `2015-summary.csv`
   - JSON: `.../flight-data/json/2015-summary.json`
3. After upload, note the path (e.g. `/FileStore/uploads/your-file.json`). Create a folder structure under **File Store** if needed (e.g. `day01/flight-data/csv/`, `day01/flight-data/json/`) and set **`BASE_PATH`** in the next cell to match (without a trailing slash).
   - Example: if your JSON is at `/FileStore/day01/flight-data/json/2015-summary.json`, use `BASE_PATH = "/FileStore/day01/flight-data"`.

In [0]:
%run ./02-Mount-Azure-Data-Lake

# Mount Azure Data Lake Storage (ADLS Gen2) in Databricks

This notebook shows how to **mount an Azure Data Lake Storage Gen2 container** so you can read and write data using DBFS paths in later labs.

---

Set your **student_id** in Step 1 below; the notebook will mount to `/mnt/data-<student_id>`. Attach this notebook to your cluster.

---

## Prerequisites

1. **Azure Data Lake Storage Gen2** (or a storage account with hierarchical namespace enabled).
2. A **container** (e.g. `inputdata`) with your data uploaded.
3. **Service Principal (SP)** in Azure AD with at least **Storage Blob Data Contributor** (or Reader) on the container/storage account.
4. **Databricks:** Secret scope with SP credentials (recommended), or use variables below for lab only (do not commit secrets).

## Step 1 — Your student ID and Data Lake account

In [0]:
# Set this to the path where you uploaded flight-data (folder containing csv/ and json/).
# Example: "/FileStore/day01/flight-data" or "/FileStore/uploads/flight-data"
BASE_PATH = base_path
OUTPUT_PATH = f"abfss://{containerName}@{adlsAccountName}.dfs.core.windows.net/data/OUTPUT"

flight_data_json = BASE_PATH + "/json/2015-summary.json"

In [0]:
df = spark.read.format("csv") \
    .option("header", True) \
    .load("file:/tmp/2010-summary.csv")


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8819763450842021>, line 3
      1 df = spark.read.format("csv") \
      2     .option("header", True) \
----> 3     .load("file:/tmp/2010-summary.csv")

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:312, in DataFrameReader.load(self, path, format, schema, **options)
    310 self.options(**options)
    311 if isinstance(path, str):
--> 312     return self._df(self._jreader.load(path))
    313 elif path is not None:
    314     if type(path) != list

In [0]:
# Create Parquet from CSV (run once). The notebook uses Parquet later; we don't ship .parquet in the repo.
csv_path = BASE_PATH + "/2010-summary.csv"
parquet_path = BASE_PATH + "/parquet"
spark.read.format("csv").option("header", True).load(csv_path).write.mode("overwrite").format("parquet").save(parquet_path)

**DataFrames**

### Understanding DataFrames

**1. What is a Schema?**
A schema defines the structure of a dataset, specifying the column names, data types, and constraints. It ensures that the data adheres to a predefined format, enabling consistency and compatibility across processing workflows.

- **Benefits of Schema:**
  - Enforces data integrity and validation.
  - Enables query optimization.
  - Facilitates compatibility with downstream applications.

*Example*: Viewing the schema of a DataFrame:

**2. DataFrames**
A DataFrame is a distributed collection of data organized into named columns, similar to a table in a relational database. It is designed to support both structured and semi-structured data processing.

- **Key Features of DataFrames:**
  - Schema-based processing.
  - Lazy evaluation for efficient execution.
  - Catalyst optimizer for query planning.
  - Multi-language support: Python, Scala, SQL, R.

*Example*: Loading a JSON dataset:

In [0]:
# Load JSON data (flight_data_json set in the Data Path cell above)
df = spark.read.format("json").load(flight_data_json)

df.show(5)

In [0]:
df.printSchema()

### File Formats in Spark

**1. JSON (JavaScript Object Notation)**
A lightweight, semi-structured format ideal for data exchange.

- **Advantages:**
  - Human-readable.
  - Flexible structure for nested data.

*Example*: Reading and writing JSON:

In [0]:
# Reading JSON
json_df = spark.read.format("json").load(BASE_PATH + "/json/2015-summary.json")

# Writing JSON
json_df.write.format("json").save(OUTPUT_PATH + "/flight-data-json")

**2. Parquet**
A columnar storage format optimized for analytics, offering efficient compression and query performance.

- **Advantages:**
  - Supports schema evolution.
  - Optimized for columnar operations.

*Example*: Working with Parquet:

In [0]:
# Reading Parquet (folder created in the "Create Parquet from CSV" cell above)
parquet_df = spark.read.format("parquet").load(BASE_PATH + "/parquet")

# Writing Parquet
parquet_df.write.format("parquet").save(OUTPUT_PATH + "/flight-data-parquet")

**3. Avro**
A row-based binary format commonly used for data serialization.

- **Advantages:**
  - Compact and efficient.
  - Supports schema definition and evolution.

*Example*: Working with Avro:

In [0]:
parquet_df.write.format("avro").save(OUTPUT_PATH + "/flight-data-avro")



In [0]:
# Reading Avro
avro_df = spark.read.format("avro").load(OUTPUT_PATH + "/flight-data-avro")
avro_df.display()

**4. CSV (Comma-Separated Values)**
A widely used format for simple tabular data.

- **Advantages:**
  - Easy to understand and use.
  - Compatible with many tools.

*Example*: Working with CSV:

In [0]:
# Reading CSV
csv_df = spark.read.format("csv").option("header", True).load(BASE_PATH + "/csv/2010-summary.csv")

# Writing CSV
csv_df.write.format("csv").option("header", True).save(OUTPUT_PATH + "/flight-data-csv")

---

### Schema Management and Optimization

**1. Inferring Schema**
Spark automatically detects column names and data types based on file contents.

*Example*: Inferring schema from JSON:

In [0]:
schema_inferred_df = spark.read.json(flight_data_json)
schema_inferred_df.printSchema()

**2. Explicit Schema Definition**
Define schemas for strict data validation and compatibility.

*Example*: Defining a schema:

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", LongType(), True)
])

schema_defined_df = spark.read.format("json").schema(schema).load(flight_data_json)
schema_defined_df.printSchema()

---



---

**Spark SQL**

### SQL in Spark: Overview and Usage

Spark SQL is a module in Apache Spark that provides a structured data processing interface using SQL-like queries. It bridges the gap between the relational and procedural paradigms, allowing data analysts and developers to interact with data using SQL while leveraging Spark's scalability and performance.

**Key Features of Spark SQL:**
- **Unified Data Access**: Access data from various sources like JSON, Parquet, Avro, Hive, and JDBC.
- **Schema-Based Processing**: Enables schema definition and enforcement for structured data.
- **Integration with DataFrames**: Allows seamless transitions between SQL queries and DataFrame APIs.
- **Optimized Query Execution**: Uses the Catalyst Optimizer for efficient query planning.

*Example 1: Running SQL queries on a DataFrame*

In [0]:
# Load data from JSON (flight_data_json and BASE_PATH set in the File Store section above)
df = spark.read.format("json").load(flight_data_json)

# Register DataFrame as a SQL table
df.createOrReplaceTempView("flight_data")

# Query using Spark SQL
query = """
SELECT DEST_COUNTRY_NAME, SUM(count) AS total_flights 
FROM flight_data 
GROUP BY DEST_COUNTRY_NAME 
ORDER BY total_flights DESC 
LIMIT 10
"""
sql_result = spark.sql(query)
sql_result.show()

---

### Creating and Managing Tables with Spark SQL

Spark SQL allows users to create and manage tables, supporting both temporary and permanent tables.

**1. Temporary Views:**
Temporary views exist only within the Spark session and are not persistent.

*Example 2: Creating and querying a temporary view*

In [0]:
# Creating a temporary view
df.createOrReplaceTempView("temp_flights")

# Querying the temporary view
query = """
SELECT ORIGIN_COUNTRY_NAME, COUNT(*) AS num_flights
FROM temp_flights
GROUP BY ORIGIN_COUNTRY_NAME
HAVING num_flights > 10
ORDER BY num_flights DESC
LIMIT 5
"""
spark.sql(query).show()

# More complex query with multiple aggregations
query_advanced = """
SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS total_passengers, AVG(count) AS avg_flights
FROM temp_flights
GROUP BY ORIGIN_COUNTRY_NAME
ORDER BY total_passengers DESC
LIMIT 10
"""
spark.sql(query_advanced).show()

**2. Global Temporary Views:**
Global temporary views are accessible across multiple sessions using the `global_temp` database.

*Example 3: Creating and querying a global temporary view*

In [0]:
# Create a global temporary view
df.createOrReplaceGlobalTempView("global_flights")

# Query the global view
query = "SELECT * FROM global_temp.global_flights WHERE count > 500"
spark.sql(query).show()

**3. Managed Tables:**
Managed tables are stored in Spark's warehouse directory and managed by Spark.

*Example 4: Creating a managed table and inserting data*

In [0]:
spark.sql("CREATE TABLE IF NOT EXISTS managed_table (DEST_COUNTRY_NAME STRING, ORIGIN_COUNTRY_NAME STRING, count LONG)")
spark.sql("INSERT INTO managed_table SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, TRY_CAST(count AS LONG) AS count FROM temp_flights")
spark.sql("SELECT * FROM managed_table").show()

---

### Performance Optimization with Catalyst Optimizer

The Catalyst Optimizer is a key component of Spark SQL, designed to enhance query performance through logical and physical plan optimization.

**Key Techniques:**
- **Predicate Pushdown:** Filters are applied early to reduce data scanning.
- **Column Pruning:** Reads only required columns from the data source.
- **Join Optimization:** Reorders joins for optimal execution.
- **Query Caching:** Uses in-memory storage for frequently accessed data.

*Example 6: Using query caching for performance optimization*

In [0]:
# Cache query result
cached_result = spark.sql("""
SELECT DEST_COUNTRY_NAME, SUM(count) AS total_flights
FROM flight_data
GROUP BY DEST_COUNTRY_NAME
""")
cached_result.cache()
cached_result.count()
cached_result.show()

---

### Using Databricks SQL for Queries and Visualization

Databricks SQL provides a user-friendly interface for running SQL queries and creating visualizations.

**Steps to Use Databricks SQL:**
1. Navigate to the SQL editor in Databricks.
2. Select or create a SQL warehouse.
3. Write and execute SQL queries.
4. Create dashboards and visualizations.

*Example 7: Running a SQL query in Databricks SQL*

In [0]:
query = """
SELECT DEST_COUNTRY_NAME, SUM(count) AS total_flights
FROM managed_table
GROUP BY DEST_COUNTRY_NAME
ORDER BY total_flights DESC
LIMIT 10
"""
spark.sql(query).show()

---

---

## Widgets — Parameterized Notebooks

Widgets let users pass values into the notebook without changing code.

**Create a widget, read its value, use in PySpark and SQL.**

In [0]:
dbutils.widgets.text("category_filter", "A")

In [0]:
category = dbutils.widgets.get("category_filter")
print(category)

Use the widget in SQL with `${widget_name}` (run the widget cell first).

In [0]:
%sql
SELECT * FROM flight_data LIMIT 5